# Fase 18 — Campanha Oficial 30R × 3 datasets × 3 cenários

Execute a célula operacional em um runtime Colab de RAM alta. A execução grava checkpoint após cada rodada e retoma automaticamente. Somente resultados com 30 rodadas, nove experimentos completos e teste avaliado apenas no final recebem `usable_in_thesis=true`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
from pathlib import Path
from collections import deque
from datetime import datetime, timezone
import csv, gc, hashlib, json, os, shutil, subprocess, sys, time, urllib.request, zipfile
subprocess.run([sys.executable,'-m','pip','install','-q','tenseal'],check=True)
import numpy as np, pandas as pd, psutil, tenseal as ts
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

CAMPAIGN_ID='THESIS_OFFICIAL_CAMPAIGN_V2_20260901'
PROTOCOL_VERSION='phase18_official_v1_common_local_sgd_hybrid_ts0125'
SEED=42; NUM_CLIENTS=5; ROUNDS=30; LOCAL_EPOCHS=3; BATCH_SIZE=64; LEARNING_RATE=0.1; THRESHOLD=0.5
TRANSPORT_SCALE=0.125
DATASETS=['PHYSIONET_CHALLENGE_2012','DAHL_RATS','CHEXCHONET']
SCENARIOS=['BASELINE','CKKS','HYBRID']
CKKS={'poly_modulus_degree':8192,'coeff_mod_bit_sizes':[60,40,40,60],'global_scale':2**40,'slots':4096,'library':'TenSEAL/Microsoft SEAL'}
BASE=Path('/content/drive/MyDrive/Mestrado_Criptografia/OFFICIAL_CAMPAIGN_V2')
ROOT=BASE/CAMPAIGN_ID; CONTROL=ROOT/'00_CAMPAIGN_CONTROL'; FREEZE=ROOT/'01_DATASET_FREEZE'; OFFICIAL=ROOT/'04_OFFICIAL_CAMPAIGN'; EXPORTS=CONTROL/'EXPORTS'
OFFICIAL.mkdir(parents=True,exist_ok=True); EXPORTS.mkdir(parents=True,exist_ok=True)
EXPECTED_PARTITION_HASHES={'PHYSIONET_CHALLENGE_2012':'d175d06e19dcb0df7c668bf2184bfb93bf9e2d76cff1d8a430ec11ffa2fb7153','DAHL_RATS':'ebaba9d08cd6a5303bc739423893dbe6d62a951e2982c33b1b53173414a69a5d','CHEXCHONET':'e149adc9da9d9498b2df9c5cd6546f0d18ac5b732ff432a510c2e9f7f9281de7'}
EXPECTED_DATA_HASHES={'PHYSIONET_CHALLENGE_2012':'9b177585e87eee8bb201e9abd451a2a0bea80776773d1c0f45b50c54d2b1e76a','DAHL_RATS':'b0539e4a6ecd9eb36e0ae18379c9cd3787f1546a248eb00b78a9e19d10b90af7','CHEXCHONET':'12c7145628e25bd389cb642c38d19f8c4e8927c8d2f5b2182d7fe215dabd60a4'}
DAHL_FEATURES=['mean','std','min','max','median','q05','q25','q75','q95','rms','range','mean_abs_diff']

def utc(): return datetime.now(timezone.utc).isoformat()
def sha256_file(path,chunk=8<<20):
 h=hashlib.sha256()
 with Path(path).open('rb') as f:
  for block in iter(lambda:f.read(chunk),b''): h.update(block)
 return h.hexdigest()
def atomic_json(path,value):
 path=Path(path); tmp=path.with_suffix(path.suffix+'.tmp'); tmp.write_text(json.dumps(value,indent=2,ensure_ascii=False),encoding='utf-8'); os.replace(tmp,path)
def atomic_npy(path,value):
 path=Path(path); tmp=path.with_suffix(path.suffix+'.tmp')
 with tmp.open('wb') as f: np.save(f,np.asarray(value),allow_pickle=False)
 os.replace(tmp,path)

print('='*110); print('FASE 18 — CAMPANHA OFICIAL 30 RODADAS × 3 DATASETS × 3 CENÁRIOS'); print('CAMPAIGN=',ROOT); print('PROTOCOL=',PROTOCOL_VERSION)
ram=psutil.virtual_memory(); disk=shutil.disk_usage('/content'); print(f'RAM total={ram.total/1024**3:.2f} GiB; disponível={ram.available/1024**3:.2f} GiB; disco livre={disk.free/1024**3:.2f} GiB')
assert ram.total>=20*1024**3 and ram.available>=16*1024**3,'A campanha oficial exige runtime de RAM alta.'

gate_path=CONTROL/'PHASE17E1_MASTER_GATE.json'; assert gate_path.exists(),f'Gate ausente: {gate_path}'
gate=json.loads(gate_path.read_text(encoding='utf-8'))
assert gate['all_hybrid_smokes_approved'] is True and gate['official_campaign_authorized'] is True
assert gate['protocol_version']=='phase17e1_hybrid_smoke_v5_fixed_transport_scale_0p125' and float(gate['transport_scale'])==TRANSPORT_SCALE
print('PHASE17E1_AUTHORIZATION_GATE=PASS')

def load_indices(path):
 z=np.load(path,allow_pickle=False); keys=list(z.files)
 def pick(names):
  for name in names:
   if name in z: return np.asarray(z[name],dtype=np.int64).reshape(-1)
  raise KeyError(f'Chaves ausentes {names}; disponíveis={keys}')
 return pick(['train','train_idx','idx_train']),pick(['validation','val','validation_idx','val_idx','idx_val']),pick(['test','test_idx','idx_test'])

def load_clients(path,dataset):
 assert sha256_file(path)==EXPECTED_PARTITION_HASHES[dataset],f'{dataset}: hash de partição divergente'
 z=np.load(path,allow_pickle=False); clients=[]
 for i in range(NUM_CLIENTS):
  for key in [f'client_{i}',f'client{i}',f'client_{i}_indices',f'client{i}_indices',str(i)]:
   if key in z: clients.append(np.asarray(z[key],dtype=np.int64).reshape(-1)); break
  else: raise KeyError(f'{dataset}: cliente {i} ausente; chaves={z.files}')
 return clients

def prepare_dataset(dataset):
 root=FREEZE/dataset/'SCIENTIFIC_FREEZE'; partition=root/'official_client_partition_noniid_seed42.npz'; clients=load_clients(partition,dataset)
 if dataset=='PHYSIONET_CHALLENGE_2012':
  data=root/'physionet_challenge_2012_features.csv'; assert sha256_file(data)==EXPECTED_DATA_HASHES[dataset]
  df=pd.read_csv(data); features=[c for c in df.columns if c not in {'RecordID','target'}]; X=df[features].apply(pd.to_numeric,errors='coerce').to_numpy(np.float64); y=pd.to_numeric(df['target']).to_numpy(np.int64); tr,va,te=load_indices(root/'official_split_seed42.npz')
 elif dataset=='DAHL_RATS':
  data=root/'dahl_derived_features.csv'; assert sha256_file(data)==EXPECTED_DATA_HASHES[dataset]
  df=pd.read_csv(data); features=DAHL_FEATURES; X=df[features].apply(pd.to_numeric,errors='coerce').to_numpy(np.float64); y=pd.to_numeric(df['target']).to_numpy(np.int64); tr,va,te=load_indices(root/'official_split_seed42.npz')
 else:
  data=root/'chexchonet_embeddings.npz'; assert sha256_file(data)==EXPECTED_DATA_HASHES[dataset]
  z=np.load(data,allow_pickle=False); X=np.asarray(z['embeddings'],dtype=np.float64); y=np.asarray(z['targets'],dtype=np.int64); features=[f'embedding_{i:03d}' for i in range(X.shape[1])]
  raw=np.load(root/'official_split_original.npy',allow_pickle=True)
  def norm(v):
   if isinstance(v,(bytes,np.bytes_)): v=v.decode('utf-8')
   s=str(v).strip().lower(); return {'0':'train','1':'validation','2':'test','val':'validation','valid':'validation','dev':'validation'}.get(s,s)
  labels=np.array([norm(v) for v in raw]); tr=np.where(labels=='train')[0]; va=np.where(labels=='validation')[0]; te=np.where(labels=='test')[0]
 preprocessing=root/'preprocessing_train_only.npz'; assert preprocessing.exists(),f'{dataset}: pré-processamento congelado ausente'
 p=np.load(preprocessing,allow_pickle=True); median=np.asarray(p['median'],np.float64); mean=np.asarray(p['mean'],np.float64); std=np.asarray(p['std'],np.float64)
 filled=np.where(np.isfinite(X),X,median); X=(filled-mean)/std
 initial_path=root/'shared_initial_state.npy'; assert initial_path.exists(),f'{dataset}: estado inicial compartilhado ausente'
 initial=np.asarray(np.load(initial_path,allow_pickle=False),np.float64).reshape(-1)
 assert len(initial)==X.shape[1]+1 and np.isfinite(X).all() and np.isfinite(initial).all()
 assert not (set(tr)&set(va) or set(tr)&set(te) or set(va)&set(te)); assert set(np.concatenate(clients))==set(tr)
 return {'root':root,'X':X,'y':y,'train':tr,'validation':va,'test':te,'clients':clients,'initial':initial,'features':features,'partition_sha256':sha256_file(partition),'preprocessing_sha256':sha256_file(preprocessing),'initial_sha256':sha256_file(initial_path),'data_sha256':sha256_file(data)}

def sigmoid(x): x=np.clip(x,-40,40); return 1/(1+np.exp(-x))
def evaluate(state,X,y,idx):
 prob=sigmoid(X[idx]@state[:-1]+state[-1]); pred=(prob>=THRESHOLD).astype(np.int64); tn,fp,fn,tp=confusion_matrix(y[idx],pred,labels=[0,1]).ravel()
 return {'accuracy':float(accuracy_score(y[idx],pred)),'precision':float(precision_score(y[idx],pred,zero_division=0)),'recall':float(recall_score(y[idx],pred,zero_division=0)),'f1':float(f1_score(y[idx],pred,zero_division=0)),'auroc':float(roc_auc_score(y[idx],prob)) if len(np.unique(y[idx]))==2 else None,'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)},prob
def local_delta(state,X,y,idx,client_id,round_number):
 w=state[:-1].copy(); b=float(state[-1]); start=time.perf_counter(); cpu0=time.process_time()
 for epoch in range(LOCAL_EPOCHS):
  order=np.asarray(idx).copy(); np.random.default_rng(SEED+100000*round_number+1000*client_id+epoch).shuffle(order)
  for s in range(0,len(order),BATCH_SIZE):
   batch=order[s:s+BATCH_SIZE]; probability=sigmoid(X[batch]@w+b); error=probability-y[batch]; w-=LEARNING_RATE*(X[batch].T@error/len(batch)); b-=LEARNING_RATE*float(error.mean())
 result=np.r_[w,b]-state
 return result,{'client_id':client_id,'samples':int(len(idx)),'train_wall_seconds':time.perf_counter()-start,'train_cpu_seconds':time.process_time()-cpu0,'delta_l2_norm':float(np.linalg.norm(result))}

def run_id(dataset,scenario): return f'RUN__{dataset}__{scenario}__OFFICIAL__R30__C5__S42'+('__TS0125' if scenario=='HYBRID' else '')
def config_for(dataset,scenario,data):
 return {'campaign_id':CAMPAIGN_ID,'protocol_version':PROTOCOL_VERSION,'dataset':dataset,'scenario':scenario,'execution_type':'OFFICIAL','usable_in_thesis_when_complete':True,'seed':SEED,'rounds':ROUNDS,'clients':NUM_CLIENTS,'local_epochs':LOCAL_EPOCHS,'batch_size':BATCH_SIZE,'learning_rate':LEARNING_RATE,'threshold':THRESHOLD,'model':'LogisticRegression','optimizer':'SGD','loss':'binary_cross_entropy','aggregation':'weighted_FedAvg','common_local_training_across_scenarios':True,'feature_count':len(data['initial'])-1,'parameter_count':len(data['initial']),'train_count':len(data['train']),'validation_count':len(data['validation']),'test_count':len(data['test']),'test_policy':'evaluated_once_only_after_round_30','partition_sha256':data['partition_sha256'],'data_sha256':data['data_sha256'],'preprocessing_sha256':data['preprocessing_sha256'],'initial_state_sha256':data['initial_sha256'],'ckks':CKKS if scenario=='CKKS' else None,'hybrid':{'transport_scale':TRANSPORT_SCALE,'rubato_variant':'RUBATO80S','bridge_commit':'105fc73115b56f1d6ff357029c7682b19a6d8510','bridge_source_sha256':'105f72d1dcb0a2a69b7fbda9b9b82de3603d1a05c9c8f11946baf75252febef6','decryptions_before_aggregation':0} if scenario=='HYBRID' else None}

def prepare_run(dataset,scenario,data):
 directory=OFFICIAL/dataset/scenario/run_id(dataset,scenario); rounds_dir=directory/'rounds'; checkpoints=directory/'checkpoints'; logs=directory/'logs'
 for path in [directory,rounds_dir,checkpoints,logs]: path.mkdir(parents=True,exist_ok=True)
 config=config_for(dataset,scenario,data); config_hash=hashlib.sha256(json.dumps(config,sort_keys=True,separators=(',',':')).encode()).hexdigest(); config['config_sha256']=config_hash
 config_path=directory/'RUN_CONFIG.json'
 if config_path.exists():
  saved=json.loads(config_path.read_text(encoding='utf-8')); assert saved.get('config_sha256')==config_hash,f'{dataset}/{scenario}: configuração anterior incompatível'
 else: atomic_json(config_path,config)
 initial_checkpoint=checkpoints/'state_round_000.npy'
 if not initial_checkpoint.exists(): atomic_npy(initial_checkpoint,data['initial'])
 completed=0
 for r in range(1,ROUNDS+1):
  if (rounds_dir/f'round_{r:03d}_metrics.json').exists() and (checkpoints/f'state_round_{r:03d}.npy').exists():
   previous=json.loads((rounds_dir/f'round_{r:03d}_metrics.json').read_text(encoding='utf-8')); assert previous.get('approved') is True,f'{dataset}/{scenario}: rodada persistida {r} não aprovada'; completed=r
  else: break
 state=np.asarray(np.load(checkpoints/f'state_round_{completed:03d}.npy',allow_pickle=False),np.float64)
 status={'run_id':run_id(dataset,scenario),'campaign_id':CAMPAIGN_ID,'dataset':dataset,'scenario':scenario,'status':'RUNNING' if completed<ROUNDS else 'ROUNDS_COMPLETED','completed_rounds':completed,'total_rounds':ROUNDS,'usable_in_thesis':False,'resumed_at_utc':utc()}
 atomic_json(directory/'RUN_STATUS.json',status)
 return directory,rounds_dir,checkpoints,logs,state,completed,config

def create_ckks_contexts():
 secret=ts.context(ts.SCHEME_TYPE.CKKS,poly_modulus_degree=CKKS['poly_modulus_degree'],coeff_mod_bit_sizes=CKKS['coeff_mod_bit_sizes']); secret.global_scale=CKKS['global_scale']; secret.generate_galois_keys(); public_bytes=secret.serialize(save_public_key=True,save_secret_key=False,save_galois_keys=True,save_relin_keys=True); return secret,ts.context_from(public_bytes),len(public_bytes)


REPO=Path('/content/RtF-Transciphering'); COMMIT='105fc73115b56f1d6ff357029c7682b19a6d8510'
subprocess.run(['apt-get','update','-qq'],check=True); subprocess.run(['apt-get','install','-y','-qq','golang-go'],check=True)
ARCHIVE_URL=f'https://codeload.github.com/KAIST-CryptLab/RtF-Transciphering/zip/{COMMIT}'; ARCHIVE_PATH=Path('/content')/f'RtF-Transciphering-{COMMIT}.zip'; EXTRACT_ROOT=Path('/content/rtf_archive_extract')
def valid_rtf_tree(path): return path.is_dir() and (path/'go.mod').is_file() and (path/'ckks_fv').is_dir()
if REPO.exists() and not valid_rtf_tree(REPO): shutil.rmtree(REPO)
if not valid_rtf_tree(REPO):
 last_error=None
 for attempt in range(1,4):
  try:
   if ARCHIVE_PATH.exists(): ARCHIVE_PATH.unlink()
   request=urllib.request.Request(ARCHIVE_URL,headers={'User-Agent':'Mozilla/5.0 Phase18 official campaign','Accept':'application/zip'})
   with urllib.request.urlopen(request,timeout=180) as response, ARCHIVE_PATH.open('wb') as out: shutil.copyfileobj(response,out,length=8<<20)
   assert ARCHIVE_PATH.stat().st_size>0
   with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    assert archive.testzip() is None
    shutil.rmtree(EXTRACT_ROOT,ignore_errors=True); EXTRACT_ROOT.mkdir(parents=True)
    for member in archive.infolist(): assert str((EXTRACT_ROOT/member.filename).resolve()).startswith(str(EXTRACT_ROOT.resolve())+os.sep)
    archive.extractall(EXTRACT_ROOT)
   candidates=[p for p in EXTRACT_ROOT.iterdir() if valid_rtf_tree(p)]; assert len(candidates)==1; shutil.move(str(candidates[0]),str(REPO)); shutil.rmtree(EXTRACT_ROOT,ignore_errors=True); break
  except Exception as error:
   last_error=error; print(f'RTF_DOWNLOAD_FAILED={attempt}/3',repr(error)); shutil.rmtree(EXTRACT_ROOT,ignore_errors=True); time.sleep(5*attempt)
 else: raise RuntimeError(f'RtF indisponível: {last_error}')
assert valid_rtf_tree(REPO)
GO_SOURCE='package ckks_fv\n\nimport (\n    "crypto/rand"\n    "encoding/json"\n    "fmt"\n    "math"\n    "os"\n    "runtime"\n    "runtime/debug"\n    "testing"\n    "time"\n    "github.com/ldsec/lattigo/v2/utils"\n)\n\ntype phase17E0Request struct {\n    Dataset string `json:"dataset"`\n    OriginalLength int `json:"original_length"`\n    ClientDeltas [][]float64 `json:"client_deltas"`\n    ClientWeights []float64 `json:"client_weights"`\n}\ntype phase17E0Response struct {\n    Dataset string `json:"dataset"`\n    OriginalLength int `json:"original_length"`\n    Clients int `json:"clients"`\n    RecoveredAggregate []float64 `json:"recovered_aggregate"`\n    MaxAbsError float64 `json:"max_abs_error"`\n    MeanAbsError float64 `json:"mean_abs_error"`\n    MaxPaddingError float64 `json:"max_padding_error"`\n    WallSeconds float64 `json:"wall_seconds"`\n    RubatoVariant string `json:"rubato_variant"`\n    BridgeVersion string `json:"bridge_version"`\n    AggregatedBeforeDecrypt bool `json:"aggregated_before_decrypt"`\n    ClientDecryptions int `json:"client_decryptions"`\n}\n\nfunc TestPhase17E0EncryptedFedAvg(t *testing.T) {\n    raw := os.Getenv("PHASE17E0_REQUEST_JSON")\n    if raw == "" { t.Fatal("PHASE17E0_REQUEST_JSON ausente") }\n    var req phase17E0Request\n    if err := json.Unmarshal([]byte(raw), &req); err != nil { t.Fatal(err) }\n    if len(req.ClientDeltas) != 5 || len(req.ClientWeights) != 5 { t.Fatal("expected exactly 5 clients and 5 weights") }\n    if req.OriginalLength < 1 || req.OriginalLength > 513 { t.Fatal("original_length must be in [1,513]") }\n    sumW := 0.0\n    for i := range req.ClientDeltas {\n        if len(req.ClientDeltas[i]) != req.OriginalLength { t.Fatalf("client %d: expected %d values, got %d", i, req.OriginalLength, len(req.ClientDeltas[i])) }\n        if req.ClientWeights[i] <= 0 { t.Fatalf("client %d: weight must be positive", i) }\n        sumW += req.ClientWeights[i]\n    }\n    if math.Abs(sumW-1.0) > 1e-12 { t.Fatalf("weights must sum to 1; got %.17g", sumW) }\n\n    rubatoParam := RUBATO80S\n    blocksize := RubatoParams[rubatoParam].Blocksize\n    numRound := RubatoParams[rubatoParam].NumRound\n    plainModulus := RubatoParams[rubatoParam].PlainModulus\n    sigma := RubatoParams[rubatoParam].Sigma\n    hbtpParams := RtFRubatoParams[0]\n    params, err := hbtpParams.Params(); if err != nil { t.Fatal(err) }\n    params.SetPlainModulus(plainModulus); params.SetLogFVSlots(params.LogN())\n    messageScaling := float64(params.PlainModulus()) / hbtpParams.MessageRatio\n    rubatoModDown := RubatoModDownParams[rubatoParam].CipherModDown\n    stcModDown := RubatoModDownParams[rubatoParam].StCModDown\n    kgen := NewKeyGenerator(params); sk, pk := kgen.GenKeyPairSparse(hbtpParams.H)\n    fvEncoder := NewMFVEncoder(params); ckksEncoder := NewCKKSEncoder(params)\n    fvEncryptor := NewMFVEncryptorFromPk(params, pk); ckksDecryptor := NewCKKSDecryptor(params, sk)\n    rotationsHalfBoot := kgen.GenRotationIndexesForHalfBoot(params.LogSlots(), hbtpParams)\n    pDcds := fvEncoder.GenSlotToCoeffMatFV(2)\n    rotations := append(rotationsHalfBoot, kgen.GenRotationIndexesForSlotsToCoeffsMat(pDcds)...)\n    rotkeys := kgen.GenRotationKeysForRotations(rotations, true, sk); rlk := kgen.GenRelinearizationKey(sk)\n    evk := EvaluationKey{Rlk: rlk, Rtks: rotkeys}\n    hbtp, err := NewHalfBootstrapper(params, hbtpParams, BootstrappingKey{Rlk: rlk, Rtks: rotkeys}); if err != nil { t.Fatal(err) }\n    fvEvaluator := NewMFVEvaluator(params, evk, pDcds)\n    ckksEvaluator := NewCKKSEvaluator(params, evk)\n    key := make([]uint64, blocksize); for i := range key { key[i] = uint64(i+1) }\n    keyRubato := NewMFVRubato(rubatoParam, params, fvEncoder, fvEncryptor, fvEvaluator, rubatoModDown[0])\n    kCt := keyRubato.EncKey(key)\n\n    start := time.Now()\n    expected := make([]float64, 513)\n    var encryptedAggregate *Ciphertext\n    for clientID := 0; clientID < 5; clientID++ {\n        fmt.Printf("PHASE17E0_PROGRESS dataset=%s client=%d/5 stage=transciphering\\n", req.Dataset, clientID+1)\n        data := make([]float64, params.N()); copy(data[:req.OriginalLength], req.ClientDeltas[clientID])\n        for j := 0; j < req.OriginalLength; j++ { expected[j] += req.ClientWeights[clientID] * req.ClientDeltas[clientID][j] }\n        nonces := make([][]byte, params.N()); keystream := make([][]uint64, params.N())\n        for i := 0; i < params.N(); i++ { nonces[i] = make([]byte,8); if _,err=rand.Read(nonces[i]);err!=nil{t.Fatal(err)} }\n        counter := make([]byte,8); if _,err=rand.Read(counter);err!=nil{t.Fatal(err)}\n        for i := 0; i < params.N(); i++ { keystream[i] = plainRubato(blocksize,numRound,nonces[i],counter,key,plainModulus,sigma) }\n        coeffs := make([]float64, params.N())\n        for i := 0; i < params.N()/2; i++ { j:=utils.BitReverse64(uint64(i),uint64(params.LogN()-1)); coeffs[j]=data[i]; coeffs[j+uint64(params.N()/2)]=data[i+params.N()/2] }\n        plainRingT := ckksEncoder.EncodeCoeffsRingTNew(coeffs,messageScaling); poly:=plainRingT.Value()[0]\n        for i := 0; i < params.N(); i++ { j:=utils.BitReverse64(uint64(i),uint64(params.LogN())); poly.Coeffs[0][j]=(poly.Coeffs[0][j]+keystream[i][0])%params.PlainModulus() }\n        plaintext:=NewPlaintextFVLvl(params,0); fvEncoder.FVScaleUp(plainRingT,plaintext)\n        // A fresh Rubato workspace per client prevents mutable modulus-chain state from leaking across clients.\n        clientRubato:=NewMFVRubato(rubatoParam,params,fvEncoder,fvEncryptor,fvEvaluator,rubatoModDown[0])\n        fvKeystreams:=clientRubato.Crypt(nonces,counter,kCt,rubatoModDown); fvKS:=fvEvaluator.SlotsToCoeffs(fvKeystreams[0],stcModDown)\n        level:=fvKS.Level()\n        if level>0 { fvEvaluator.ModSwitchMany(fvKS,fvKS,level) }\n        ciphertext:=NewCiphertextFVLvl(params,1,0); ciphertext.Value()[0]=plaintext.Value()[0].CopyNew(); fvEvaluator.Sub(ciphertext,fvKS,ciphertext); fvEvaluator.TransformToNTT(ciphertext,ciphertext)\n        ciphertext.SetScale(math.Exp2(math.Round(math.Log2(float64(params.Qi()[0])/float64(params.PlainModulus())*messageScaling))))\n        ctBoot,_:=hbtp.HalfBoot(ciphertext,false)\n        weighted:=ckksEvaluator.MultByConstNew(ctBoot,req.ClientWeights[clientID])\n        if encryptedAggregate==nil { encryptedAggregate=weighted } else { ckksEvaluator.Add(encryptedAggregate,weighted,encryptedAggregate) }\n        fmt.Printf("PHASE17E0_PROGRESS dataset=%s client=%d/5 stage=encrypted_aggregate_updated\\n", req.Dataset, clientID+1)\n        // Release all client-local material before the next transciphering. Only the CKKS aggregate survives.\n        data=nil; nonces=nil; keystream=nil; coeffs=nil; plainRingT=nil; plaintext=nil\n        clientRubato=nil; fvKeystreams=nil; fvKS=nil; ciphertext=nil; ctBoot=nil; weighted=nil\n        runtime.GC(); debug.FreeOSMemory()\n        fmt.Printf("PHASE17E0_PROGRESS dataset=%s client=%d/5 stage=memory_released\\n", req.Dataset, clientID+1)\n    }\n    // SECURITY BOUNDARY: the sole decrypt operation is after all five CKKS ciphertexts were aggregated.\n    values:=ckksEncoder.DecodeComplex(ckksDecryptor.DecryptNew(encryptedAggregate),params.LogSlots())\n    recovered:=make([]float64,req.OriginalLength); maxErr:=0.0; meanErr:=0.0; maxPad:=0.0\n    for i:=0;i<req.OriginalLength;i++ { recovered[i]=real(values[i]); e:=math.Abs(recovered[i]-expected[i]); meanErr+=e; if e>maxErr{maxErr=e} }\n    meanErr/=float64(req.OriginalLength)\n    for i:=req.OriginalLength;i<513;i++ { e:=math.Abs(real(values[i])); if e>maxPad{maxPad=e} }\n    resp:=phase17E0Response{req.Dataset,req.OriginalLength,5,recovered,maxErr,meanErr,maxPad,time.Since(start).Seconds(),"RUBATO80S","phase17e0_encrypted_fedavg_v1",true,0}\n    b,err:=json.Marshal(resp);if err!=nil{t.Fatal(err)};fmt.Printf("PHASE17E0_JSON:%s\\n",b)\n}\n'
GO_FILE=REPO/'ckks_fv'/'phase17e0_encrypted_fedavg_test.go'; GO_FILE.write_text(GO_SOURCE,encoding='utf-8')
assert hashlib.sha256(GO_SOURCE.encode()).hexdigest()=='105f72d1dcb0a2a69b7fbda9b9b82de3603d1a05c9c8f11946baf75252febef6'
subprocess.run(['gofmt','-w',str(GO_FILE)],check=True); compile_gate=subprocess.run(['go','test','./ckks_fv','-run','^$','-count=1'],cwd=REPO,text=True,capture_output=True); print(compile_gate.stdout,compile_gate.stderr); assert compile_gate.returncode==0
print('HYBRID_BRIDGE_GATE=PASS')


def aggregate_baseline(deltas,weights):
 n=len(deltas[0]); communication={'model_download_bytes':NUM_CLIENTS*n*8,'client_update_bytes':NUM_CLIENTS*n*8,'total_bytes':2*NUM_CLIENTS*n*8}
 return np.average(np.stack(deltas),axis=0,weights=weights),{'aggregation_seconds':0.0,'communication':communication,'numeric_error':{'max_abs_error':0.0,'mean_abs_error':0.0,'rmse':0.0},'checks':{'weighted_fedavg':True}}

def aggregate_ckks(deltas,weights,secret,public,context_bytes):
 encrypted=[]; rows=[]; start=time.perf_counter()
 for delta in deltas:
  t=time.perf_counter(); ct=ts.ckks_vector(public,delta.tolist()); blob=ct.serialize(); rows.append({'encryption_seconds':time.perf_counter()-t,'ciphertext_bytes':len(blob)}); encrypted.append(ct)
 aggregate=None; agg_start=time.perf_counter()
 for ct,weight in zip(encrypted,weights): aggregate=ct*float(weight) if aggregate is None else aggregate+ct*float(weight)
 aggregation_seconds=time.perf_counter()-agg_start; blob=aggregate.serialize(); dec_start=time.perf_counter(); recovered=np.asarray(ts.ckks_vector_from(secret,blob).decrypt(),np.float64)[:len(deltas[0])]; decryption_seconds=time.perf_counter()-dec_start
 clear=np.average(np.stack(deltas),axis=0,weights=weights); error=np.abs(recovered-clear); numeric={'max_abs_error':float(error.max()),'mean_abs_error':float(error.mean()),'rmse':float(np.sqrt(np.mean(error**2)))}
 communication={'public_context_bytes':context_bytes,'client_ciphertexts_bytes':int(sum(r['ciphertext_bytes'] for r in rows)),'aggregate_ciphertext_bytes':len(blob),'total_bytes':int(context_bytes+sum(r['ciphertext_bytes'] for r in rows)+len(blob))}
 checks={'finite':bool(np.isfinite(recovered).all()),'max_abs_error_le_1e_3':numeric['max_abs_error']<=1e-3,'mean_abs_error_le_1e_4':numeric['mean_abs_error']<=1e-4,'aggregated_before_decrypt':True,'client_decryptions_zero':True}
 return recovered,{'wall_seconds':time.perf_counter()-start,'aggregation_seconds':aggregation_seconds,'decryption_seconds':decryption_seconds,'clients_crypto':rows,'communication':communication,'numeric_error':numeric,'checks':checks}

def aggregate_hybrid(dataset,round_number,run_dir,deltas,weights):
 scaled=[np.asarray(delta*TRANSPORT_SCALE,np.float64) for delta in deltas]; clear=np.average(np.stack(deltas),axis=0,weights=weights); equivalence=float(np.max(np.abs(np.average(np.stack(scaled),axis=0,weights=weights)/TRANSPORT_SCALE-clear))); assert equivalence<=1e-12
 request={'dataset':dataset,'original_length':len(clear),'client_deltas':[x.tolist() for x in scaled],'client_weights':weights.tolist()}; env=os.environ.copy(); env['PHASE17E0_REQUEST_JSON']=json.dumps(request,separators=(',',':')); env['GOGC']='20'; env.pop('GOMEMLIMIT',None)
 local_log=Path('/content')/f'{run_id(dataset,"HYBRID")}__round_{round_number:03d}.log'; drive_log=run_dir/'logs'/f'round_{round_number:03d}_bridge.log'; payload=None; tail=deque(maxlen=100); start=time.perf_counter()
 with local_log.open('w',encoding='utf-8') as out:
  process=subprocess.Popen(['go','test','./ckks_fv','-run','^TestPhase17E0EncryptedFedAvg$','-count=1','-v','-timeout=0'],cwd=REPO,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
  for line in process.stdout:
   line=line.rstrip(); out.write(line+'\n'); out.flush(); tail.append(line)
   if 'PHASE17E0_PROGRESS' in line: print(line,flush=True)
   if 'PHASE17E0_JSON:' in line: payload=json.loads(line.split('PHASE17E0_JSON:',1)[1])
  return_code=process.wait()
 shutil.copy2(local_log,drive_log)
 if return_code!=0 or payload is None: raise RuntimeError(f'{dataset} rodada {round_number}: bridge falhou rc={return_code}; log={drive_log}; cauda={list(tail)}')
 recovered=np.asarray(payload['recovered_aggregate'],np.float64)/TRANSPORT_SCALE; error=np.abs(recovered-clear); nonbias=float(error[:-1].max()) if len(error)>1 else 0.0; bias_abs=float(error[-1]); bias_relative=bias_abs/max(abs(float(clear[-1])),1e-12); relative_l2=float(np.linalg.norm(recovered-clear)/max(np.linalg.norm(clear),1e-12)); padding=float(payload['max_padding_error'])/TRANSPORT_SCALE
 numeric={'max_abs_error':float(error.max()),'nonbias_max_abs_error':nonbias,'bias_abs_error':bias_abs,'bias_relative_error':bias_relative,'global_relative_l2_error':relative_l2,'mean_abs_error':float(error.mean()),'rmse':float(np.sqrt(np.mean(error**2))),'max_padding_error':padding,'transport_equivalence_error':equivalence}
 checks={'finite':bool(np.isfinite(recovered).all()),'nonbias_max_abs_error_le_1e_3':nonbias<=1e-3,'bias_abs_le_1e_3_or_relative_le_2e_3':bias_abs<=1e-3 or bias_relative<=2e-3,'relative_l2_le_2e_3_or_rmse_le_1e_4':relative_l2<=2e-3 or numeric['rmse']<=1e-4,'mean_abs_error_le_1e_4':numeric['mean_abs_error']<=1e-4,'max_padding_error_le_1e_3':padding<=1e-3,'aggregated_before_decrypt':payload['aggregated_before_decrypt'] is True,'client_decryptions_zero':payload['client_decryptions']==0,'transport_equivalence_le_1e_12':equivalence<=1e-12}
 communication={'logical_model_download_bytes':NUM_CLIENTS*len(clear)*8,'logical_client_update_bytes':NUM_CLIENTS*len(clear)*8,'total_bytes':2*NUM_CLIENTS*len(clear)*8,'measurement_scope':'logical payload equivalent; no network serialization in local bridge'}
 return recovered,{'wall_seconds':time.perf_counter()-start,'bridge_wall_seconds':payload['wall_seconds'],'communication':communication,'numeric_error':numeric,'checks':checks,'bridge_log':str(drive_log)}

all_results=[]
for dataset in DATASETS:
 data=prepare_dataset(dataset)
 for scenario in SCENARIOS:
  print('='*110); print(f'OFFICIAL_START dataset={dataset} scenario={scenario} rounds={ROUNDS} clients={NUM_CLIENTS}')
  run_dir,rounds_dir,checkpoints,logs,state,completed,config=prepare_run(dataset,scenario,data)
  if (run_dir/'FINAL_RESULT.json').exists():
   final=json.loads((run_dir/'FINAL_RESULT.json').read_text(encoding='utf-8')); assert final['approved'] is True; print('OFFICIAL_RESULT_REUSED=',final['run_id']); all_results.append(final); continue
  secret=public=context_bytes=None
  if scenario=='CKKS': secret,public,context_bytes=create_ckks_contexts()
  for round_number in range(completed+1,ROUNDS+1):
   round_start=time.perf_counter(); cpu_start=time.process_time(); child_start=__import__('resource').getrusage(__import__('resource').RUSAGE_CHILDREN); rss_start=psutil.Process().memory_info().rss; deltas=[]; client_rows=[]
   for client_id,indices in enumerate(data['clients']):
    delta,row=local_delta(state,data['X'],data['y'],indices,client_id,round_number); assert np.isfinite(delta).all(); deltas.append(delta); client_rows.append(row)
   weights=np.asarray([len(x) for x in data['clients']],np.float64); weights/=weights.sum(); clear=np.average(np.stack(deltas),axis=0,weights=weights)
   if scenario=='BASELINE': recovered,crypto=aggregate_baseline(deltas,weights)
   elif scenario=='CKKS': recovered,crypto=aggregate_ckks(deltas,weights,secret,public,context_bytes)
   else:
    gc.collect()
    try: __import__('ctypes').CDLL('libc.so.6').malloc_trim(0)
    except Exception: pass
    recovered,crypto=aggregate_hybrid(dataset,round_number,run_dir,deltas,weights)
   next_state=state+recovered; validation,validation_prob=evaluate(next_state,data['X'],data['y'],data['validation']); clear_prob=sigmoid(data['X'][data['validation']]@(state+clear)[:-1]+(state+clear)[-1]); probability_difference=float(np.max(np.abs(validation_prob-clear_prob))); disagreement=float(np.mean((validation_prob>=THRESHOLD)!=(clear_prob>=THRESHOLD)))
   checks=dict(crypto['checks']); checks.update({'five_clients_completed':len(client_rows)==5,'finite_state':bool(np.isfinite(next_state).all()),'partition_hash_preserved':data['partition_sha256']==EXPECTED_PARTITION_HASHES[dataset],'test_not_used':True,'validation_probability_difference_le_1e_3':probability_difference<=1e-3,'validation_prediction_disagreement_le_1e_3':disagreement<=1e-3}); approved=all(checks.values())
   child_end=__import__('resource').getrusage(__import__('resource').RUSAGE_CHILDREN)
   metrics_row={'campaign_id':CAMPAIGN_ID,'run_id':run_id(dataset,scenario),'dataset':dataset,'scenario':scenario,'round':round_number,'approved':approved,'usable_in_thesis':False,'started_at_utc':utc(),'wall_seconds':time.perf_counter()-round_start,'python_cpu_seconds':time.process_time()-cpu_start,'child_cpu_seconds':float((child_end.ru_utime+child_end.ru_stime)-(child_start.ru_utime+child_start.ru_stime)),'rss_start_bytes':rss_start,'rss_end_bytes':psutil.Process().memory_info().rss,'validation':validation,'validation_probability_max_abs_difference_from_clear_round':probability_difference,'validation_prediction_disagreement_from_clear_round':disagreement,'numeric_error':crypto['numeric_error'],'communication':crypto['communication'],'crypto':{k:v for k,v in crypto.items() if k not in {'numeric_error','communication','checks'}},'clients':client_rows,'checks':checks}
   atomic_json(rounds_dir/f'round_{round_number:03d}_metrics.json',metrics_row); pd.DataFrame(client_rows).to_csv(rounds_dir/f'round_{round_number:03d}_clients.csv',index=False); atomic_npy(checkpoints/f'state_round_{round_number:03d}.npy',next_state)
   atomic_json(run_dir/'RUN_STATUS.json',{'run_id':run_id(dataset,scenario),'campaign_id':CAMPAIGN_ID,'dataset':dataset,'scenario':scenario,'status':'RUNNING' if round_number<ROUNDS else 'ROUNDS_COMPLETED','completed_rounds':round_number,'total_rounds':ROUNDS,'last_round_approved':approved,'usable_in_thesis':False,'updated_at_utc':utc()})
   print(f'OFFICIAL_PROGRESS dataset={dataset} scenario={scenario} round={round_number}/{ROUNDS} approved={approved} val_auroc={validation["auroc"]} wall={metrics_row["wall_seconds"]:.3f}s',flush=True)
   assert approved,f'{dataset}/{scenario}/round_{round_number:03d} reprovada; Fase 18 interrompida'
   state=next_state; del deltas; gc.collect()
  final_validation,_=evaluate(state,data['X'],data['y'],data['validation']); final_test,_=evaluate(state,data['X'],data['y'],data['test']); round_files=[rounds_dir/f'round_{r:03d}_metrics.json' for r in range(1,ROUNDS+1)]; round_records=[json.loads(p.read_text(encoding='utf-8')) for p in round_files]
  final={'campaign_id':CAMPAIGN_ID,'protocol_version':PROTOCOL_VERSION,'run_id':run_id(dataset,scenario),'run_dir':str(run_dir),'dataset':dataset,'scenario':scenario,'execution_type':'OFFICIAL','rounds':ROUNDS,'clients':NUM_CLIENTS,'approved':all(r['approved'] for r in round_records),'usable_in_thesis':all(r['approved'] for r in round_records),'test_evaluated_only_after_round_30':True,'validation':final_validation,'test':final_test,'total_wall_seconds':float(sum(r['wall_seconds'] for r in round_records)),'total_communication_bytes':int(sum((r['communication'] or {}).get('total_bytes',0) for r in round_records)),'communication_interpretation':'Baseline=logical serialized float64 payload; CKKS=serialized ciphertext plus context as measured per round; Hybrid=logical payload equivalent because bridge is local without network serialization','completed_at_utc':utc(),'config_sha256':config['config_sha256'],'round_metrics_sha256':{p.name:sha256_file(p) for p in round_files},'final_state_sha256':sha256_file(checkpoints/'state_round_030.npy')}
  atomic_json(run_dir/'FINAL_RESULT.json',final); atomic_json(run_dir/'RUN_STATUS.json',{'run_id':final['run_id'],'campaign_id':CAMPAIGN_ID,'dataset':dataset,'scenario':scenario,'status':'COMPLETED_APPROVED' if final['approved'] else 'COMPLETED_REJECTED','completed_rounds':ROUNDS,'test_evaluated':True,'usable_in_thesis':final['usable_in_thesis'],'completed_at_utc':utc()}); assert final['approved']; all_results.append(final); print('OFFICIAL_COMPLETED=',final['run_id'])
 gc.collect()

all_ok=len(all_results)==9 and all(r['approved'] and r['usable_in_thesis'] for r in all_results)
comparison=pd.DataFrame([{'dataset':r['dataset'],'scenario':r['scenario'],'rounds':r['rounds'],'validation_accuracy':r['validation']['accuracy'],'validation_f1':r['validation']['f1'],'validation_auroc':r['validation']['auroc'],'test_accuracy':r['test']['accuracy'],'test_precision':r['test']['precision'],'test_recall':r['test']['recall'],'test_f1':r['test']['f1'],'test_auroc':r['test']['auroc'],'total_wall_seconds':r['total_wall_seconds'],'total_communication_bytes':r['total_communication_bytes'],'usable_in_thesis':r['usable_in_thesis'],'run_dir':r['run_dir']} for r in all_results])
comparison_path=OFFICIAL/'PHASE18_COMPARISON.csv'; comparison.to_csv(comparison_path,index=False)
master={'phase':'18','campaign_id':CAMPAIGN_ID,'protocol_version':PROTOCOL_VERSION,'completed_at_utc':utc(),'official_campaign_completed':all_ok,'results_usable_in_thesis':all_ok,'matrix':'3 datasets x 3 scenarios x 30 rounds x 5 clients','test_policy':'test evaluated once after round 30','results':all_results,'comparison_csv':str(comparison_path),'next_authorized_step':'PHASE18_FINAL_REPORT_AND_DISSERTATION_UPDATE' if all_ok else None}
atomic_json(CONTROL/'PHASE18_MASTER_GATE.json',master)
status_path=CONTROL/'CAMPAIGN_STATUS.json'; status=json.loads(status_path.read_text(encoding='utf-8')); status.update({'status':'PHASE18_COMPLETED' if all_ok else 'PHASE18_INCOMPLETE','phase18_official_campaign_completed':all_ok,'results_usable_in_thesis':all_ok,'next_authorized_step':master['next_authorized_step']}); atomic_json(status_path,status)
evidence=Path('/content/PHASE18_EVIDENCE'); shutil.rmtree(evidence,ignore_errors=True); evidence.mkdir(); shutil.copy2(CONTROL/'PHASE18_MASTER_GATE.json',evidence/'PHASE18_MASTER_GATE.json'); shutil.copy2(comparison_path,evidence/'PHASE18_COMPARISON.csv'); shutil.copy2(status_path,evidence/'CAMPAIGN_STATUS.json')
for result in all_results:
 src=Path(result['run_dir']); dst=evidence/result['dataset']/result['scenario']; dst.mkdir(parents=True); shutil.copy2(src/'RUN_CONFIG.json',dst/'RUN_CONFIG.json'); shutil.copy2(src/'RUN_STATUS.json',dst/'RUN_STATUS.json'); shutil.copy2(src/'FINAL_RESULT.json',dst/'FINAL_RESULT.json'); shutil.copytree(src/'rounds',dst/'rounds'); shutil.copytree(src/'logs',dst/'logs'); (dst/'checkpoints').mkdir(); shutil.copy2(src/'checkpoints'/'state_round_030.npy',dst/'checkpoints'/'state_round_030.npy')
archive=Path(shutil.make_archive('/content/PHASE18_EVIDENCE','zip','/content','PHASE18_EVIDENCE')); permanent=EXPORTS/'PHASE18_EVIDENCE.zip'; shutil.copy2(archive,permanent)
print('='*110); print(json.dumps({'PHASE18_APPROVED':all_ok,'results_usable_in_thesis':all_ok,'experiments_completed':len(all_results),'gate':str(CONTROL/'PHASE18_MASTER_GATE.json'),'comparison':str(comparison_path),'evidence_zip':str(permanent),'next_authorized_step':master['next_authorized_step']},indent=2)); print('='*110); assert all_ok


## Retomada

Se o Colab desconectar, abra este mesmo notebook novamente e execute a célula. Cada execução retomará do maior checkpoint consecutivo validado. Não apague `04_OFFICIAL_CAMPAIGN`.
